# KazGPT V6 — Максимальный датасет + Обучение на A100

**Источники данных:**
- 📚 issai/kazqad — QA от ISSAI/Назарбаев Университет
- 🌐 Wikipedia (kk) — Казахская Википедия
- 🔷 mC4 (kk) — Веб-корпус
- ✨ CulturaX (kk) — Очищенный веб
- 📖 OSCAR-2301 (kk) — Common Crawl
- 🗂️ multidomain-kazakh — Мульти-доменный корпус
- 🌍 Aya Dataset (kk) — Диалоги
- 💬 Alpaca EN→KK — Переведённые инструкции
- ✍️ Manual seeds — Высококачественные примеры

**Перед запуском:**
1. `Runtime → Change runtime type → A100 GPU`
2. Смонтируй Google Drive (ячейка 1)
3. Вставь HF токен в ячейку 2
4. Запусти все ячейки по порядку

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 1 — Монтирование Drive + установка зависимостей
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/KazGPT_V6'
DATA_PATH  = f'{DRIVE_PATH}/data'
for p in [DATA_PATH, f'{DRIVE_PATH}/checkpoints',
          f'{DRIVE_PATH}/adapter', f'{DRIVE_PATH}/gguf']:
    os.makedirs(p, exist_ok=True)
print(f'✅ Drive смонтирован: {DRIVE_PATH}')

!pip install -q datasets huggingface_hub requests beautifulsoup4 lxml tqdm deep-translator
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers accelerate peft trl bitsandbytes sentencepiece
print('✅ Зависимости установлены')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 2 — Конфигурация
# ═══════════════════════════════════════════════════════════════

# ⚠️ ВСТАВЬ СВОЙ ТОКЕН: https://huggingface.co/settings/tokens
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX'

# ── Лимиты на источник (уменьши для теста) ───────────────────
LIMITS = {
    'kazqad':       99999,   # ~16k примеров (весь датасет)
    'wikipedia':    80000,
    'mc4':          50000,
    'culturax':     50000,
    'oscar':        40000,
    'multidomain':  30000,
    'aya':          10000,
    'alpaca_trans': 5000,    # переводим только 5k (долго)
}

FAST_MODE = False  # True = 3000 на каждый источник (тест ~20 мин)
if FAST_MODE:
    LIMITS = {k: 3000 for k in LIMITS}
    print('⚡ FAST MODE — по 3000 на источник')

OUT_RAW    = f'{DATA_PATH}/kazgpt_v6_raw.jsonl'
OUT_TRAIN  = f'{DATA_PATH}/train.jsonl'
OUT_VALID  = f'{DATA_PATH}/valid.jsonl'

print(f'Конфиг OK. FAST={FAST_MODE}')
print(f'Данные → {DATA_PATH}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 3 — Хелперы (качество, дедупликация, форматы)
# ═══════════════════════════════════════════════════════════════
import json, re, hashlib, random, time
from datetime import datetime
from collections import Counter

random.seed(42)
STATS = Counter()

KK_CHARS = set('әғқңөұүһіӘҒҚҢӨҰҮҺІ')

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=True)

def clean(text):
    if not text: return ''
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'http\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def kk_ratio(text):
    alpha = [c for c in text if c.isalpha()]
    if not alpha: return 0.0
    return sum(1 for c in alpha if c in KK_CHARS) / len(alpha)

def is_quality(text, min_len=60, min_words=10, min_kk=0.03):
    text = text.strip()
    if len(text) < min_len: return False
    if len(text.split()) < min_words: return False
    if kk_ratio(text) < min_kk: return False
    if re.search(r'<[a-z]{1,10}[\s/>]', text, re.I): return False
    return True

def has_repetition(text, ngram=5, max_rep=3):
    words = text.split()
    if len(words) < ngram * 2: return False
    ngrams = [' '.join(words[i:i+ngram]) for i in range(len(words)-ngram+1)]
    return Counter(ngrams).most_common(1)[0][1] > max_rep

def to_rec(prompt, completion, source):
    return {'prompt': prompt.strip(),
            'completion': ' ' + completion.strip(),
            'source': source}

def fp(r):
    text = (r['prompt'] + r['completion']).lower()
    return hashlib.md5(re.sub(r'\s+', ' ', text).encode()).hexdigest()

def dedup(records):
    seen = set()
    out = []
    for r in records:
        h = fp(r)
        if h not in seen:
            seen.add(h)
            out.append(r)
    return out

def save_batch(records, name):
    path = f'{DATA_PATH}/{name}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    STATS[name] = len(records)
    log(f'  ✓ {name}: {len(records):,} примеров')
    return records

print('✅ Хелперы загружены')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 4 — issai/kazqad
# ═══════════════════════════════════════════════════════════════
from datasets import load_dataset

log('📚 [1/8] issai/kazqad ...')
kazqad_records = []
try:
    ds = load_dataset('issai/kazqad', split='train')
    for item in ds:
        q = item.get('question') or item.get('instruction') or ''
        a = item.get('answers') or item.get('answer') or ''
        if isinstance(a, dict): a = (a.get('text') or [''])[0]
        if isinstance(a, list): a = a[0] if a else ''
        q, a = clean(str(q)), clean(str(a))
        if q and len(a) > 20:
            kazqad_records.append(to_rec(q, a, 'kazqad'))
        if len(kazqad_records) >= LIMITS['kazqad']: break
    save_batch(kazqad_records, 'kazqad')
except Exception as e:
    log(f'  ✗ {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 5 — Wikipedia (kk)
# ═══════════════════════════════════════════════════════════════
log('🌐 [2/8] Wikipedia (kk) ...')
wiki_records = []
try:
    ds = load_dataset('wikipedia', '20231101.kk', split='train', streaming=True)
    templates = [
        '{t} деген не?',
        '{t} туралы қысқаша айтып бер.',
        '{t} жайында мәлімет бер.',
        '{t} дегеніміз не?',
    ]
    for item in ds:
        title = clean(item.get('title', ''))
        text  = clean(item.get('text', ''))
        if not title or not is_quality(text, min_len=200): continue
        sents = [s.strip() for s in text.split('.') if len(s.strip()) > 40]
        if len(sents) >= 3:
            summary = '. '.join(sents[:3]) + '.'
            wiki_records.append(to_rec(
                random.choice(templates).format(t=title), summary, 'wikipedia_kk'))
        words = text.split()
        if len(words) > 60:
            cut = len(words) // 3
            wiki_records.append(to_rec(
                ' '.join(words[:cut]),
                ' '.join(words[cut:cut+120]),
                'wikipedia_kk_completion'))
        if len(wiki_records) >= LIMITS['wikipedia']: break
    save_batch(wiki_records, 'wikipedia_kk')
except Exception as e:
    log(f'  ✗ {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 6 — mC4 + CulturaX + OSCAR-2301 (kk)
# ═══════════════════════════════════════════════════════════════

def load_completion_dataset(hf_name, config, split, text_field,
                             source_name, limit):
    records = []
    try:
        log(f'  Загрузка {hf_name} ...')
        ds = load_dataset(hf_name, config, split=split,
                          streaming=True, trust_remote_code=True)
        for item in ds:
            text = clean(item.get(text_field) or '')
            if not is_quality(text, min_len=150): continue
            words = text.split()
            if len(words) < 40: continue
            cut = min(len(words) // 2, 100)
            records.append(to_rec(
                ' '.join(words[:cut]),
                ' '.join(words[cut:cut+150]),
                source_name))
            if len(records) >= limit: break
        save_batch(records, source_name)
    except Exception as e:
        log(f'  ✗ {hf_name}: {e}')
    return records

log('🔷 [3/8] mC4 (kk) ...')
mc4_records = load_completion_dataset(
    'mc4', 'kk', 'train', 'text', 'mc4_kk', LIMITS['mc4'])

log('✨ [4/8] CulturaX (kk) ...')
culturax_records = load_completion_dataset(
    'uonlp/CulturaX', 'kk', 'train', 'text', 'culturax_kk', LIMITS['culturax'])

log('📖 [5/8] OSCAR-2301 (kk) ...')
oscar_records = load_completion_dataset(
    'oscar-corpus/OSCAR-2301', 'kk', 'train', 'content', 'oscar_kk', LIMITS['oscar'])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 7 — multidomain-kazakh + Aya (kk)
# ═══════════════════════════════════════════════════════════════

log('🗂️ [6/8] multidomain-kazakh ...')
multi_records = load_completion_dataset(
    'kz-transformers/multidomain-kazakh-dataset', None,
    'train', 'text', 'multidomain_kk', LIMITS['multidomain'])

log('🌍 [7/8] Aya Dataset (kk) ...')
aya_records = []
try:
    ds = load_dataset('CohereForAI/aya_dataset', split='train')
    kk_ds = ds.filter(lambda x: x.get('language') in ('kaz', 'Kazakh', 'kk'))
    for item in kk_ds:
        q = clean(item.get('inputs') or '')
        a = clean(item.get('targets') or '')
        if q and len(a) > 20:
            aya_records.append(to_rec(q, a, 'aya_kk'))
        if len(aya_records) >= LIMITS['aya']: break
    save_batch(aya_records, 'aya_kk')
except Exception as e:
    log(f'  ✗ Aya: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 8 — Alpaca EN→KK перевод (Google Translate)
# ⏱️ ~1.5с на пример → 5000 примеров ≈ 2.5 часа
#    Результат сохраняется с checkpoint (можно перезапустить)
# ═══════════════════════════════════════════════════════════════
from deep_translator import GoogleTranslator
import time

ALPACA_LIMIT = LIMITS['alpaca_trans']
ALPACA_OUT   = f'{DATA_PATH}/alpaca_kk.jsonl'
DELAY        = 1.5  # секунды между запросами

log(f'💬 [8/8] Alpaca EN→KK (лимит: {ALPACA_LIMIT}) ...')

# Checkpoint — сколько уже переведено
done = 0
if os.path.exists(ALPACA_OUT):
    with open(ALPACA_OUT) as f:
        done = sum(1 for l in f if l.strip())
    log(f'  ♻️  Checkpoint: {done} уже переведено')

try:
    ds = load_dataset('tatsu-lab/alpaca', split='train')
    tr = GoogleTranslator(source='en', target='kk')

    records_to_translate = []
    for item in ds:
        instr = (item.get('instruction') or '').strip()
        inp   = (item.get('input') or '').strip()
        out   = (item.get('output') or '').strip()
        if not instr or not out: continue
        if inp: instr = f'{instr}\n\nInput: {inp}'
        records_to_translate.append({'instruction': instr, 'output': out})
        if len(records_to_translate) >= ALPACA_LIMIT: break

    random.shuffle(records_to_translate)
    to_process = records_to_translate[done:]
    log(f'  Осталось перевести: {len(to_process)}')

    alpaca_written = done
    batch = []

    for i, item in enumerate(to_process, 1):
        try:
            p_kk = tr.translate(item['instruction'])
            time.sleep(DELAY)
            c_kk = tr.translate(item['output'])
            time.sleep(DELAY)
            if p_kk and c_kk:
                batch.append(to_rec(p_kk, c_kk, 'alpaca_translated'))
        except Exception:
            time.sleep(3)
            continue

        if len(batch) >= 50:
            with open(ALPACA_OUT, 'a', encoding='utf-8') as f:
                for r in batch:
                    f.write(json.dumps(r, ensure_ascii=False) + '\n')
            alpaca_written += len(batch)
            batch = []
            log(f'  💾 Alpaca: {alpaca_written}/{ALPACA_LIMIT}')

    if batch:
        with open(ALPACA_OUT, 'a', encoding='utf-8') as f:
            for r in batch:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        alpaca_written += len(batch)

    log(f'  ✓ alpaca_kk: {alpaca_written} примеров')
    STATS['alpaca_kk'] = alpaca_written

except Exception as e:
    log(f'  ✗ Alpaca: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 9 — Ручные seed примеры (x5 вес)
# ═══════════════════════════════════════════════════════════════
log('✍️  Manual seeds ...')

seeds = [
    ('Сәлем!',              'Сәлем! 😊 Сізге қалай көмектесе аламын?'),
    ('Сәлеметсіз бе!',      'Сәлеметсіз! Сізге қалай көмектесейін?'),
    ('Қалайсыз?',           'Жақсымын, рахмет! Сіз қалайсыз?'),
    ('Қайырлы таң!',        'Қайырлы таң! 🌅 Бүгін қалай көмектесе аламын?'),
    ('Кешке жарық!',        'Кешке жарық! Сізге қандай сұрақ бар?'),
    ('Привет!',             'Сәлем! Қазақ тілінде сөйлесейік — қалайсыз? 😊'),
    ('Hello!',              'Сәлем! KazGPT-ке қош келдіңіз 🇰🇿'),
    ('Рахмет!',             'Өтінемін! 😊 Басқа сұрағыңыз болса айтыңыз.'),
    ('Сау бол!',            'Сау болыңыз! 👋 Тағы кіріңіз.'),
    ('Өзің туралы айтып бер',
     'Мен KazGPT V6 — Қыдырбек Елдос жасаған локальды жасанды интеллект. '
     'QLoRA әдісімен Qwen2.5-7B моделін 300k+ казахша мысалдармен дообучал.'),
    ('Қазақстан туралы 3 қызықты факт айт',
     '1. Қазақстан — әлемдегі ең үлкен жерге байланысты мемлекет (2,7 млн км²).\n'
     '2. Байқоңыр ғарыш айлағы — адамзат тарихындағы алғашқы ғарыш айлағы.\n'
     '3. Қазақстанда 130-дан астам ұлт тұрады.'),
    ('Машиналық оқыту деген не?',
     'Машиналық оқыту — компьютерлерді мысалдардан үйренуге мүмкіндік беретін '
     'жасанды интеллект саласы. Модель деректерден заңдылықтарды тауып, болжам жасайды.'),
    ('Нейрондық желі қалай жұмыс істейді?',
     'Нейрондық желі — адам миын үлгілейтін алгоритм. Кіріс деректері қабаттар '
     'арқылы өтіп, салмақтар реттеледі. Жаттығу барысында қателер азайтылып, дәлдік артады.'),
    ('LoRA fine-tuning дегеніміз не?',
     'LoRA — үлкен модельдерді аз параметрмен дообучать ету әдісі. '
     'Барлық салмақтарды емес, арнайы қосымша матрицаларды ғана өзгертеді. Жылдам, арзан, тиімді.'),
    ('Python-да Фибоначчи санын есептейтін функция жаз',
     'def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)'),
    ('Python-да 1-ден 10-ға дейін санды экранға шығар',
     'for i in range(1, 11):\n    print(i)'),
    ('Алматы туралы мәлімет бер',
     'Алматы — Қазақстанның ең үлкен қаласы. Іле Алатауының солтүстік '
     'баурайында орналасқан. Халқы 2 миллионнан астам. 1997 жылға дейін астана болған.'),
    ('Қазақстан қай жылы тәуелсіздік алды?',
     'Қазақстан 1991 жылы 16 желтоқсанда тәуелсіздік жариялады.'),
    ('Неліктен аспан көк?',
     'Аспан Рэлей шашырауы салдарынан көк болып көрінеді: қысқа толқынды '
     'көк сәуле атмосферада көп шашырайды.'),
    ('Жасанды интеллект пен машиналық оқытудың айырмашылығы?',
     'Жасанды интеллект — кеңірек ұғым, машиналардың ақылды мінез-құлқы. '
     'Машиналық оқыту — AI-дың бір бөлімі, мысалдардан үйрену арқылы жұмыс істейді.'),
]

seed_records = [to_rec(q, a, 'manual') for q, a in seeds] * 5  # x5 вес
save_batch(seed_records, 'manual_seeds')
log(f'  ✓ {len(seeds)} seeds × 5 = {len(seed_records)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 10 — Мерж, фильтрация, дедупликация, сплит
# ═══════════════════════════════════════════════════════════════
import json

log('🔧 Финальная сборка ...')

# Собираем все источники из файлов (надёжнее чем из памяти)
all_records = []
for fname in os.listdir(DATA_PATH):
    if fname.endswith('.jsonl') and fname != 'train.jsonl' and fname != 'valid.jsonl':
        with open(f'{DATA_PATH}/{fname}', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try: all_records.append(json.loads(line))
                    except: pass

log(f'  Загружено: {len(all_records):,}')

# Фильтрация
good = []
bad = Counter()
for r in all_records:
    c = (r.get('completion') or '').strip()
    if len(c) < 20:        bad['too_short'] += 1; continue
    if len(c) > 6000:      c = c[:6000]
    if has_repetition(c):  bad['repetition'] += 1; continue
    punct = sum(1 for ch in c if ch in '!?.,;:')
    if len(c) > 0 and punct / len(c) > 0.3: bad['punct_spam'] += 1; continue
    r['completion'] = ' ' + c.strip()
    good.append(r)

log(f'  После фильтра: {len(good):,} (отброшено: {dict(bad)})')

# Дедупликация
before = len(good)
good = dedup(good)
log(f'  После дедупликации: {len(good):,} (удалено дублей: {before - len(good):,})')

# Сохранить raw
with open(OUT_RAW, 'w', encoding='utf-8') as f:
    for r in good:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
log(f'  ✓ Raw: {OUT_RAW}')

# Train/Valid сплит
random.shuffle(good)
n_valid = max(200, int(len(good) * 0.02))
train = good[n_valid:]
valid = good[:n_valid]

for path, data in [(OUT_TRAIN, train), (OUT_VALID, valid)]:
    with open(path, 'w', encoding='utf-8') as f:
        for r in data:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

log('\n' + '═'*50)
log('✅ ДАТАСЕТ ГОТОВ')
log('═'*50)
log(f'  Всего    : {len(good):,}')
log(f'  Train    : {len(train):,}')
log(f'  Valid    : {len(valid):,}')
log('═'*50)
log('\nПо источникам:')
src_count = Counter(r.get('source','?') for r in good)
for src, cnt in src_count.most_common():
    bar = '█' * min(30, cnt // max(1, max(src_count.values()) // 30))
    log(f'  {src:<35} {cnt:>8,}  {bar}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 11 — Обучение KazGPT V6 на A100
# ═══════════════════════════════════════════════════════════════
import torch
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
log(f'GPU: {gpu} | VRAM: {vram:.0f}GB')

MODEL_ID   = 'Qwen/Qwen2.5-7B-Instruct'
MAX_SEQ    = 2048
LORA_R     = 64
LORA_ALPHA = 128
BATCH      = 4 if vram >= 40 else 2
GRAD_ACC   = 4
EPOCHS     = 3
LR         = 2e-4

SYSTEM = ('Сен — KazGPT, Қазақстан үшін жасалған жергілікті жасанды интеллект. '
          'АБСОЛЮТТІ ЕРЕЖЕ: Барлық жауаптарыңды ТЕК ҚАЗАҚ ТІЛІНДЕ жаз. '
          'Сыпайы, кәсіби бол. Эмодзиді тек қуаныш/сәлем сөздерінде қолдан.')

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID, max_seq_length=MAX_SEQ, load_in_4bit=True, token=HF_TOKEN)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)

model.print_trainable_parameters()
log('✅ Модель загружена, LoRA применён')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 12 — SFTTrainer + старт обучения
# ═══════════════════════════════════════════════════════════════
import time, json
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

def load_jsonl_as_chatml(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line.strip())
            text = (
                f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
                f'<|im_start|>user\n{r["prompt"]}<|im_end|>\n'
                f'<|im_start|>assistant\n{r["completion"].strip()}<|im_end|>'
            )
            records.append({'text': text})
    return records

train_ds = Dataset.from_list(load_jsonl_as_chatml(OUT_TRAIN))
valid_ds = Dataset.from_list(load_jsonl_as_chatml(OUT_VALID))
log(f'Train: {len(train_ds):,} | Valid: {len(valid_ds):,}')

bf16 = torch.cuda.is_bf16_supported()
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=valid_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ, packing=True,
    args=TrainingArguments(
        output_dir=f'{DRIVE_PATH}/checkpoints',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACC,
        warmup_ratio=0.05, learning_rate=LR,
        lr_scheduler_type='cosine',
        fp16=not bf16, bf16=bf16,
        evaluation_strategy='epoch',
        save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        logging_steps=50, report_to='none', seed=42,
    ),
)

log('🚀 Обучение начинается ...')
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 3600
log(f'✅ Обучение завершено за {elapsed:.2f}ч | best_eval_loss={trainer.state.best_metric:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ЯЧЕЙКА 13 — Сохранение адаптера + экспорт GGUF
# ═══════════════════════════════════════════════════════════════
adapter_dir = f'{DRIVE_PATH}/adapter'
gguf_dir    = f'{DRIVE_PATH}/gguf'

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
log(f'✅ Адаптер: {adapter_dir}')

log('📦 Экспорт в GGUF Q4_K_M (5-15 мин) ...')
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method='q4_k_m')

import glob
files = glob.glob(f'{gguf_dir}/*.gguf')
for f in files:
    size = os.path.getsize(f) / 1e9
    log(f'  GGUF: {os.path.basename(f)} ({size:.1f} GB)')

log('\n✅ V6 полностью готов!')
log(f'Скачай GGUF из Google Drive → KazGPT_V6/gguf/')

## После обучения — подключение KazGPT V6

### 1. Скачай GGUF из Google Drive
Папка: `KazGPT_V6/gguf/` → файл `*.gguf` (~4–5 GB)

### 2. Создай Modelfile
```
FROM /путь/к/kazgpt-v6.gguf

PARAMETER temperature 0.27
PARAMETER top_p 0.85
PARAMETER repeat_penalty 1.15
PARAMETER num_predict 512

SYSTEM "Сен — KazGPT, Қазақстан үшін жасалған жергілікті жасанды интеллект. Барлық жауаптарыңды ТЕК ҚАЗАҚ ТІЛІНДЕ жаз."
```

### 3. Зарегистрируй в Ollama
```bash
ollama create kazgpt-v6 -f Modelfile
ollama run kazgpt-v6 "Сәлем!"
```

### 4. Обнови application.yml
```yaml
kazgpt:
  models:
    base:
      name: kazgpt-v6
```